In [457]:
# Paquetes y librerías
library(readr)
library(dplyr)
library(ggplot2)
library(viridisLite)
library(here)  # para rutas relativas
# install.packages("alakazam")
library(alakazam)
library(ineq)
library(vegan)
library(tidyr)
library(tibble)

## Carga de datos y cálculo de abundancias clonales

En esta sección se carga un archivo `.tsv` que contiene un repertorio 
simulado de secuencias de inmunoglobulinas.

Posteriormente:

- Se agrega un identificador de muestra (`sample_id`), ya que el repertorio es simulado.
- Se agrupan las secuencias por clon (`clone_id`) y muestra.
- Se calcula el número de secuencias por clon, generando una tabla de **abundancias clonales (`clone_counts`)**.

Esta tabla será utilizada posteriormente para el cálculo de métricas de diversidad clonotípica.

In [458]:

# Cargar tu archivo .tsv
archivo_clones <- "../data/output/repertorio_D_insilico_25600_seqs_clone-pass.tsv"
clones <- read_tsv(archivo_clones)
# Agregar identificador de muestra (porque es simulado)
clones <- clones %>%
  mutate(sample_id = "repertorio_simulado")


# Contar secuencias por clon y muestra
clone_counts <- clones %>%
  group_by(sample_id, clone_id) %>%
  summarise(count = n(), .groups = "drop")

Rows: 25599 Columns: 50
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (20): sequence_id, sequence, v_call, d_call, j_call, sequence_alignment,...
dbl (25): junction_length, np1_length, np2_length, v_sequence_start, v_seque...
lgl  (5): rev_comp, productive, stop_codon, vj_in_frame, c_call

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Cálculo de números de Hill

La función `hill_numbers()` calcula los números de Hill 
para valores de q desde 0 hasta 4 utilizando la función 
`calcDiversity()`.

Para ello, se define un rango de valores de q (`0:4`) y se 
utiliza la función `sapply()` para aplicar `calcDiversity()` 
a cada valor de q de forma iterativa.

Finalmente, los resultados se almacenan en una tabla con 
dos columnas:

- `q`: orden de diversidad  
- `d`: valor de diversidad (número de Hill)

La función retorna esta tabla (`hill_table`), la cual se 
utiliza posteriormente para extraer métricas específicas 
como richness (`q=0`), Shannon (`q=1`) y Simpson (`q=2`).

In [459]:

hill_numbers <- function(clone_counts){

    q_values <- 0:4

    diversity_values <- sapply(q_values, function(q) {
        calcDiversity(clone_counts$count, q)
    })

    hill_table <- data.frame(
        q = q_values,
        d = diversity_values
    )

    return(hill_table)
}

# Ejecutar
rep_hill_numbers <- hill_numbers(clone_counts)

rep_hill_numbers

q,d
<int>,<dbl>
0,16710.0000
1,7878.2867
2,803.9701
3,270.1539
4,170.7218


## Números de Hill como métricas unificadas de diversidad

Los números de Hill representan un marco unificado para el cálculo 
de métricas clásicas de diversidad, permitiendo expresar distintas 
medidas de diversidad dentro de una misma escala: el **número efectivo 
de clones**.

Este enfoque permite integrar métricas clásicas como richness, 
Shannon y Simpson dentro de un mismo sistema, facilitando la 
comparación directa entre repertorios clonales.

A partir de la tabla de números de Hill (`rep_hill_numbers`), 
se extraen valores específicos de diversidad correspondientes 
a distintos órdenes de diversidad (`q`). Para cada métrica, 
se filtra la tabla según el valor de `q` y se extrae el valor 
de diversidad (`d`).

### Métricas clásicas representadas por los números de Hill

- **q = 0 → Richness (riqueza clonal)**  
  Representa el número total de clones únicos presentes 
  en el repertorio. No considera la abundancia relativa 
  de los clones.

- **q = 1 → Shannon (exp(H), número efectivo de clones)**  
  Corresponde al exponencial del índice de Shannon clásico, 
  considerando la frecuencia relativa de los clones.

- **q = 2 → Simpson (dominancia clonal)**  
  Da mayor peso a los clones más abundantes, permitiendo 
  evaluar la dominancia dentro del repertorio.

- **q = 3 y q = 4 → Órdenes superiores de Hill**  
  Incrementan progresivamente el peso de los clones dominantes, 
  permitiendo evaluar estructuras de dominancia más marcadas.

Los valores obtenidos para cada orden de diversidad (`q`) 
se utilizan posteriormente para comparar la diversidad 
clonotípica entre repertorios simulados y diferentes 
condiciones experimentales.

In [460]:
richness <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 0) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

richness(rep_hill_numbers)

[1] 16710

In [461]:
 d1 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d1(rep_hill_numbers)

[1] 7878.287

In [462]:
shannon <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(log(metric_value))
} 

shannon(rep_hill_numbers)

[1] 8.971866

In [463]:
 d2 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d2(rep_hill_numbers)

[1] 803.9701

In [464]:
simpson <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value))
} 

simpson(rep_hill_numbers)

[1] 0.001243827

In [465]:
 d3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 3) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d3(rep_hill_numbers)

[1] 270.1539

In [466]:
 d4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d4(rep_hill_numbers)

[1] 170.7218

## Cálculo de estimadores de riqueza Chao1 y ACE

Se calcularon estimadores no paramétricos de riqueza clonal 
utilizando la función `estimateR()` del paquete **vegan**.

Los estimadores **Chao1** y **ACE** permiten estimar la riqueza 
clonal real del repertorio considerando la presencia de clones 
raros (por ejemplo, clones observados una o pocas veces), los 
cuales pueden no estar completamente representados en la muestra.

Las abundancias clonales (`count`) se agruparon por `sample_id`, 
y posteriormente se calcularon los siguientes estimadores:

- **Chao1**: estima el número total de clones esperados, 
  considerando la frecuencia de clones raros.

- **ACE (Abundance-based Coverage Estimator)**: estima la 
  riqueza clonal basándose en la abundancia de clones poco 
  frecuentes y el grado de cobertura del muestreo.

Los valores obtenidos se almacenaron en una tabla 
(`metricas_chao1ace`) para su posterior análisis 
comparativo entre repertorios.

In [467]:
# MÉTRICA CHAO1 y ACE PAQUETE VEGAN
metricas_chao1ace <- clone_counts %>%
  group_by(sample_id) %>% 
  summarise(
    chao1 = estimateR(count)["S.chao1"],
    ace   = estimateR(count)["S.ACE"]
  )

print(metricas_chao1ace)

# A tibble: 1 x 3
  sample_id             chao1     ace
  <chr>                 <dbl>   <dbl>
1 repertorio_simulado 119151. 140267.


In [468]:


chao1 <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_chao1 <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      chao1 = as.numeric(vegan::estimateR(count)["S.chao1"]),
      .groups = "drop"
    )%>%
    dplyr::pull(chao1)
  
  return(metricas_chao1[1])
}

# Ejecutar
rep_chao1 <- chao1(clones)
print(rep_chao1)


[1] 119150.9


In [469]:
ace <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_ace <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      ace = as.numeric(vegan::estimateR(count)["S.ACE"]),
      .groups = "drop"
    )%>%
    dplyr::pull(ace)
  
  return(metricas_ace[1])
}

# Ejecutar
rep_ace <- ace(clones)
print(rep_ace)

[1] 140266.6


## Cálculo del índice de Gini (desigualdad clonal)

Se calculó el índice de Gini utilizando la función `ineq()` 
del paquete **ineq**, con el fin de evaluar el grado de 
desigualdad en la distribución de abundancias clonales.

El índice de Gini mide la dominancia clonal dentro del 
repertorio, indicando qué tan concentradas están las 
secuencias en unos pocos clones abundantes.

Para ello, se utilizó una función personalizada (`calc_gini`) 
que aplica `ineq()` a las abundancias clonales (`count`). 
Posteriormente, los datos se agruparon por `sample_id` y 
se calculó el valor de Gini para cada muestra.

### Interpretación del índice de Gini

- **Gini ≈ 0** → distribución uniforme de clones  
  (abundancias similares entre clones)

- **Gini ≈ 1** → alta desigualdad clonal  
  (pocos clones dominan el repertorio)

Los valores obtenidos se almacenaron en una tabla 
(`gini_result`) para su posterior análisis comparativo 
entre repertorios clonales.

In [470]:
# MÉTRICA GINI PAQUETE INEQ
calc_gini <- function(df) {
  ineq::ineq(df$count, type = "Gini")
}
gini_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    gini = calc_gini(cur_data())
  )

print(gini_result)

# A tibble: 1 x 2
  sample_id            gini
  <chr>               <dbl>
1 repertorio_simulado 0.335


In [471]:
gini <- function (clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    metrica_gini <- ineq::ineq(clone_counts$count, type = "Gini")
    return(metrica_gini)
}
gini(clones)

[1] 0.3347837

## Cálculo del índice de uniformidad de Pielou (evenness)

Se calculó el índice de uniformidad de **Pielou** utilizando 
funciones del paquete **vegan**, con el fin de evaluar qué 
tan uniforme es la distribución de abundancias clonales 
dentro del repertorio.

Para ello, se utilizó una función personalizada (`calc_pielou`) 
que calcula el índice de Shannon (`diversity()`) y el número 
total de clones (`specnumber()`), a partir de las abundancias 
clonales (`count`).

El índice de Pielou se calcula como:

**J = H / log(S)**

donde:

- **H** corresponde al índice de Shannon  
- **S** corresponde al número total de clones (richness)  
- **J** representa la uniformidad en la distribución clonal

Posteriormente, los datos se agruparon por `sample_id` y se 
calculó el valor de Pielou para cada muestra.

### Interpretación del índice de Pielou

- **J ≈ 1** → distribución uniforme de clones  
  (abundancias similares entre clones)

- **J ≈ 0** → baja uniformidad  
  (presencia de clones dominantes)

Los valores obtenidos se almacenaron en una tabla 
(`pielou_result`) para su posterior análisis comparativo 
entre repertorios clonales.

In [472]:
# MÉTRICA PIELOU PAQUETE VEGAN

calc_pielou <- function(df) {
  abund <- df$count
  H <- diversity(abund, index = "shannon")  # Shannon
  S <- specnumber(abund)                    # número de clones
  J <- H / log(S)                           # Pielou
  return(J)
}

pielou_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    pielou = calc_pielou(cur_data())
  )

print(pielou_result)


# A tibble: 1 x 2
  sample_id           pielou
  <chr>                <dbl>
1 repertorio_simulado  0.923


In [473]:
pielou <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  H <- vegan::diversity(clone_counts$count, index = "shannon")
  S <- vegan::specnumber(clone_counts$count)
  J <- H / log(S)
  
  return(as.numeric(J))  # 👈 devuelve solo el número
}

pielou(clones)

[1] 0.9226574

## Cálculo del índice de Basharin (corrección de Shannon)

Se calculó el índice de **Basharin**, una variante corregida 
del índice de Shannon que incorpora un término de ajuste 
para reducir el sesgo asociado a tamaños de muestra finitos.

Para ello, se utilizó una función personalizada (`calc_basharin`) 
que calcula primero el índice de Shannon (`diversity()`) y el 
número total de clones (`specnumber()`), a partir de las 
abundancias clonales (`count`).

El índice de Basharin se calcula como:

**B = H + (S − 1) / (2N)**

donde:

- **H** corresponde al índice de Shannon  
- **S** corresponde al número total de clones (richness)  
- **N** corresponde al número total de secuencias  
- **B** corresponde al índice de Basharin corregido

Este término adicional permite ajustar la estimación de 
diversidad cuando el número de secuencias es limitado, 
reduciendo el sesgo en la estimación de Shannon.

Los cálculos se realizaron agrupando los datos por 
`sample_id`, obteniendo un valor de Basharin para 
cada repertorio analizado.

Los valores obtenidos se almacenaron en la tabla 
(`basharin_result`) para su posterior análisis 
comparativo entre repertorios clonales.

In [474]:
# MÉTRICA BASHARIN FUNCIONES R+ VEGAN

calc_basharin <- function(df) {
  abund <- df$count
  N <- sum(abund)
  S <- specnumber(abund)
  
  # Casos triviales
  if (N == 0 || S <= 1) return(0)
  
  H <- diversity(abund, index = "shannon")
  print(H)
  basharin <- H + (S - 1) / (2 * N)
  return(basharin)
}

# Aplicar por muestra
basharin_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(basharin = calc_basharin(cur_data()), .groups = "drop")

print(basharin_result)


[1] 8.971701
# A tibble: 1 x 2
  sample_id           basharin
  <chr>                  <dbl>
1 repertorio_simulado     9.30


In [475]:
basharin <- function(clones_df){
  
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  abund <- clone_counts$count
  N <- sum(abund)
  S <- vegan::specnumber(abund)
  
  if (N == 0 || S <= 1) return(0)
  
  # Shannon
  H <- vegan::diversity(abund, index = "shannon")
  
  # Basharin
  basharin_val <- H + (S - 1) / (2 * N)
  
  return(as.numeric(basharin_val))  # 👈 devuelve número puro
}
basharin(clones)

[1] 9.298062

## Cálculo del índice D50 (dominancia clonal)

Se calculó el índice **D50**, una métrica utilizada para 
evaluar la dominancia clonal dentro del repertorio.

El índice D50 representa el **número mínimo de clones más 
abundantes necesarios para alcanzar el 50% del total de 
secuencias** en el repertorio.

Para su cálculo, las abundancias clonales (`count`) se 
ordenaron de mayor a menor, y posteriormente se calculó 
la suma acumulativa de las abundancias. El valor de D50 
corresponde al número de clones necesarios hasta alcanzar 
el 50% del total de secuencias.

### Interpretación del índice D50

- **D50 bajo** → alta dominancia clonal  
  (pocos clones representan gran parte del repertorio)

- **D50 alto** → mayor diversidad clonal  
  (se requieren muchos clones para alcanzar el 50%)

El valor obtenido se almacenó en una tabla (`d50_result`) 
para su posterior análisis comparativo entre repertorios 
clonales.

In [476]:
d50_fun <- function(counts) {
  counts <- sort(counts, decreasing = TRUE)
  total <- sum(counts)
  cum <- cumsum(counts)
  which(cum >= 0.5 * total)[1]
}

# Calcular D50
d50_val <- d50_fun(clone_counts$count)
print(d50_val)
d50_result <- tibble::tibble(D50 = d50_val)


[1] 3911


In [477]:
d50 <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  counts <- sort(clone_counts$count, decreasing = TRUE)
  total  <- sum(counts)
  cum    <- cumsum(counts)
  
  d50_val <- which(cum >= 0.5 * total)[1]
  
  return(as.numeric(d50_val))  # 👈 devuelve número puro
}
d50(clones)

[1] 3911

## Integración de métricas de diversidad

Se combinaron múltiples métricas de diversidad clonotípica 
(números de Hill, estimadores de riqueza y métricas de 
dominancia y uniformidad) en una única estructura 
(`metricas_diversidad`) para facilitar su análisis posterior.

In [478]:
metricas_diversidad <- c(richness= richness(rep_hill_numbers),d1= d1(rep_hill_numbers),shannon= shannon(rep_hill_numbers), d2= d2(rep_hill_numbers), simpson= simpson(rep_hill_numbers), d3= d3(rep_hill_numbers),
d4= d4(rep_hill_numbers), chao1= chao1(clones), ace= ace(clones), gini= gini(clones), pielou= pielou(clones), basharin= basharin(clones), d50= d50(clones))
metricas_diversidad


richness           d1      shannon           d2      simpson           d3 
1.671000e+04 7.878287e+03 8.971866e+00 8.039701e+02 1.243827e-03 2.701539e+02 
          d4        chao1          ace         gini       pielou     basharin 
1.707218e+02 1.191509e+05 1.402666e+05 3.347837e-01 9.226574e-01 9.298062e+00 
         d50 
3.911000e+03

## Construcción de la tabla final de diversidad

Las métricas de diversidad calculadas se integraron en una 
tabla (`tabla_diversidad`) y se añadieron variables 
descriptivas del repertorio analizado.

Se incorporaron los siguientes metadatos:

- `sample_id`: identificador del repertorio  
- `escenario`: condición o escenario simulado  
- `size`: tamaño del repertorio (número de secuencias)

Esta tabla final permite organizar las métricas y facilitar 
su comparación entre distintos escenarios y tamaños 
de repertorios simulados.

In [479]:
tabla_diversidad <- bind_rows(metricas_diversidad)
tabla_diversidad$sample_id <- "D25600seq"
tabla_diversidad$condition <- "D"
tabla_diversidad$size <- 25600

tabla_diversidad <- tabla_diversidad %>%
  select(sample_id, condition, size, everything())


In [480]:
readr::write_tsv(tabla_diversidad, "../results/diversity_metrics/diversity_D_25600seqs.tsv")